In [1]:
#AutoGptQ=python library(huggingface) for GPTQ

In [2]:
%pip install -v gptqmodel --no-build-isolation

Using pip 26.0.1 from c:\Users\INMOR14\AppData\Local\Programs\Python\Python311\Lib\site-packages\pip (python 3.11)
  Using cached gptqmodel-5.7.0.tar.gz (668 kB)
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Link requires a different Python (3.11.0 not in: '>=3.7,<3.11'): https://files.pythonhosted.org/packages/3a/be/650f9c091ef71cb01d735775d554e068752d3ff63d7943b26316dc401749/numpy-1.21.2.zip (from https://pypi.org/simple/numpy/) (requires-python:>=3.7,<3.11)
  Link requires a different Python (3.11.0 not in: '>=3.7,<3.11'): https://files.pythonhosted.org/packages/5f/d6/ad58ded26556eaeaa8c971e08b6466f17c4ac4d786cd3d800e26ce59cc01/numpy-1.21.3.zip (from https://pypi.org/simple/numpy/) (requires-python:>=3.7,<3.11)
  Link requires a different Python (3.11.0 not in: '>=3.7,<3.11'): https://files.pythonhosted.org/packages/fb/48/b0708ebd7718a8933f0d3937513ef8ef2f4f04529f1f66ca86d873043921/numpy-1.21.4.zip (from https://p

  Running command Preparing metadata (pyproject.toml)
  NVCC not found (checked PATH, $CUDA_HOME/bin, $CUDA_PATH/bin, /usr/local/cuda/bin). For Ubuntu, run `sudo update-alternatives --config cuda` to fix path for already installed Cuda.
  NVCC not found (checked PATH, $CUDA_HOME/bin, $CUDA_PATH/bin, /usr/local/cuda/bin). For Ubuntu, run `sudo update-alternatives --config cuda` to fix path for already installed Cuda.
  CUDA None
  HAS_CUDA_V8 True
  HAS_CUDA_V9 False
  SETUP_KWARGS {}
  gptqmodel_version=5.7.0
  running dist_info
  creating C:\Users\INMOR14\AppData\Local\Temp\pip-modern-metadata-ays0zwql\GPTQModel.egg-info
  writing C:\Users\INMOR14\AppData\Local\Temp\pip-modern-metadata-ays0zwql\GPTQModel.egg-info\PKG-INFO
  writing dependency_links to C:\Users\INMOR14\AppData\Local\Temp\pip-modern-metadata-ays0zwql\GPTQModel.egg-info\dependency_links.txt
  writing entry points to C:\Users\INMOR14\AppData\Local\Temp\pip-modern-metadata-ays0zwql\GPTQModel.egg-info\entry_points.txt
  wri

In [2]:
%pip install "protobuf<6.30"

Note: you may need to restart the kernel to use updated packages.


In [ ]:
%pip uninstall wheel setuptools -y
%pip install --upgrade pip
%pip install --upgrade setuptools wheel

In [14]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

# small model
model_id = "EleutherAI/gpt-neo-125M"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

# small calibration text (WikiText)
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
calibration_texts = dataset["text"][:512]  # small sample


In [15]:
from gptqmodel import GPTQModel
from gptqmodel import QuantizeConfig

In [16]:
quant_config=QuantizeConfig(bits=4,group_size=128)

INFO  QuantizeConfig: offload_to_disk_path auto set to `./gptqmodel_offload/tnyjihzb-otaxzywi/`


In [17]:
model=GPTQModel.load(model_id,quant_config)

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

INFO  Estimated Quantization BPW (bits per weight): 4.2875 bpw, based on [bits: 4, group_size: 128]


INFO  Loader: Auto dtype (CPU + Torch Fused): `torch.bfloat16`                 


model: GPTNeoForCausalLM  (P=0 B=0) [- | - | ~0B]
├─ model.transformer: GPTNeoModel  (P=0 B=0) [- | - | ~0B]
│  ├─ model.transformer.wte: Embedding  (P=38.60M B=0) [meta | bfloat16 | ~0B (est~73.62MB)]
│  │  │  param: weight  shape=(50257, 768) dtype=bfloat16 device=meta ~73.62MB
│  ├─ model.transformer.wpe: Embedding  (P=1.57M B=0) [meta | bfloat16 | ~0B (est~3.00MB)]
│  │  │  param: weight  shape=(2048, 768) dtype=bfloat16 device=meta ~3.00MB
│  ├─ model.transformer.drop: Dropout  (P=0 B=0) [- | - | ~0B]
│  ├─ model.transformer.h: ModuleList  (P=0 B=0) [- | - | ~0B]
│  │  ├─ model.transformer.h.0: GPTNeoBlock  (P=0 B=0) [- | - | ~0B]
│  │  │  ├─ model.transformer.h.0.ln_1: LayerNorm  (P=1.54K B=0) [meta | bfloat16 | ~0B (est~3.00KB)]
│  │  │  │  │  param: weight  shape=(768,) dtype=bfloat16 device=meta ~1.50KB
│  │  │  │  │  param: bias  shape=(768,) dtype=bfloat16 device=meta ~1.50KB
│  │  │  ├─ model.transformer.h.0.attn: GPTNeoAttention  (P=0 B=0) [- | - | ~0B]
│  │  │  │  └─ mode

INFO:tokenicer.tokenicer:Tokenicer: Auto fixed pad_token_id=50256 (token='<|endoftext|>').


INFO  Model: Loaded `generation_config`: GenerationConfig {
  "bos_token_id": 50256,
  "eos_token_id": 50256
}



INFO  Model: Auto-fixed `generation_config` mismatch between model and `generation_config.json`.


INFO  Model: Updated `generation_config`: GenerationConfig {
  "bos_token_id": 50256,
  "do_sample": true,
  "eos_token_id": 50256
}



INFO  Kernel: loaded -> `[]`                                                   


In [18]:
calibration_texts = [
    "Hello world!",
    "Quantization aware training is useful for LLMs.",
    "GPTQ allows 4-bit model quantization."
]

In [19]:
model.quantize(calibration=calibration_texts) 

INFO  | process | layer | module                    | feat: in, out | dtype: size  | loss          | samples | damp    | time  | fwd_time | (v)ram       | dynamic |


INFO  +---------+-------+---------------------------+---------------+--------------+---------------+---------+---------+-------+----------+--------------+---------+


INFO  | gptq    | 6     | attn.attention.q_proj     | 768, 768      | bf16: 1.2MB  | 0.0198468701  | 11      | 0.05000 | 0.642 | 0.011    | n/a          |         |


INFO  +---------+-------+---------------------------+---------------+--------------+---------------+---------+---------+-------+----------+--------------+---------+


INFO  | gptq    | 6     | attn.attention.v_proj     | 768, 768      | bf16: 1.2MB  | 0.0318627357  | 11      | 0.05000 | 0.889 | 0.011    | n/a          |         |


...

INFO  | Forward hook       | 72    | 0.010  | 0.005 | 0.330   | 0.3%   | transformer.h.11.mlp.c_proj                     |


INFO  +--------------------+-------+--------+-------+---------+--------+-------------------------------------------------+


INFO  | Capture inputs     | 1     | 0.037  | 0.037 | 0.037   | 0.0%   | cache_inputs:GPTNeoBlock                        |


INFO  +--------------------+-------+--------+-------+---------+--------+-------------------------------------------------+


{'gptq': [{'process': 'gptq',
   'layer': 0,
   'module': 'attn.attention.q_proj',
   'feat: in, out': '768, 768',
   'dtype: size': 'bf16: 1.2MB',
   'loss': '0.0397976556',
   'samples': '11',
   'damp': '0.05000',
   'time': '0.517',
   'fwd_time': '0.009',
   '(v)ram': 'n/a'},
  {'process': 'gptq',
   'layer': 0,
   'module': 'attn.attention.v_proj',
   'feat: in, out': '768, 768',
   'dtype: size': 'bf16: 1.2MB',
   'loss': '0.0245264660',
   'samples': '11',
   'damp': '0.05000',
   'time': '0.852',
   'fwd_time': '0.009',
   '(v)ram': 'n/a'},
  {'process': 'gptq',
   'layer': 0,
   'module': 'attn.attention.k_proj',
   'feat: in, out': '768, 768',
   'dtype: size': 'bf16: 1.2MB',
   'loss': '0.0396641168',
   'samples': '11',
   'damp': '0.05000',
   'time': '0.874',
   'fwd_time': '0.009',
   '(v)ram': 'n/a'},
  {'process': 'gptq',
   'layer': 0,
   'module': 'attn.attention.out_proj',
   'feat: in, out': '768, 768',
   'dtype: size': 'bf16: 1.2MB',
   'loss': '0.3592159531',
 

In [20]:
# model.quantize(calibration_texts,batch_size=1)

In [25]:
model.save(r"C:\Users\INMOR14\OneDrive - ABB\Documents\finetuning\EleutherAI\gpt-neo-125M-4bit")

INFO  Saved Quantize Config: 
{
  "bits": 4,
  "group_size": 128,
  "desc_act": false,
  "lm_head": false,
  "quant_method": "gptq",
  "checkpoint_format": "gptq",
  "pack_dtype": "int32",
  "meta": {
    "quantizer": [
      "gptqmodel:5.7.0"
    ],
    "uri": "https://github.com/modelcloud/gptqmodel",
    "damp_percent": 0.05,
    "damp_auto_increment": 0.01,
    "static_groups": false,
    "true_sequential": true,
    "mse": 0.0,
    "gptaq": null,
    "act_group_aware": true,
    "failsafe": {
      "strategy": "rtn",
      "threshold": "0.5%",
      "smooth": {
        "type": "mad",
        "group_size_threshold": 128,
        "k": 2.75
      }
    },
    "offload_to_disk": true,
    "offload_to_disk_path": "./gptqmodel_offload/tnyjihzb-otaxzywi/",
    "pack_impl": "cpu",
    "mock_quantization": false,
    "gc_mode": "interval",
    "wait_for_submodule_finalizers": false,
    "auto_forward_data_parallel": true,
    "hessian": {
      "chunk_size": null,
      "chunk_bytes": null

Files in directory:
config.json
generation_config.json
quantize_config.json
quant_log.csv
Content of saved `generation_config.json`:
{
    "_from_model_config": true,
    "bos_token_id": 50256,
    "do_sample": true,
    "eos_token_id": 50256,
    "transformers_version": "4.57.6"
}
Content of saved `config.json`:
{
    "activation_function": "gelu_new",
    "architectures": [
        "GPTNeoForCausalLM"
    ],
    "attention_dropout": 0,
    "attention_layers": [
        "global",
        "local",
        "global",
        "local",
        "global",
        "local",
        "global",
        "local",
        "global",
        "local",
        "global",
        "local"
    ],
    "attention_types": [
        [
            [
                "global",
                "local"
            ],
            6
        ]
    ],
    "bos_token_id": 50256,
    "classifier_dropout": 0.1,
    "dtype": "bfloat16",
    "embed_dropout": 0,
    "eos_token_id": 50256,
    "gradient_checkpointing": false,


INFO  Module: Re-tied embedding weights on shell model after full sync         


INFO  Module: Total synced modules: 0                                          


INFO  Pre-Quantized model size: 1003.26MB, 0.98GB                              


...

INFO  +--------------------+-------+--------+-------+---------+--------+-----------------------------------------------------------------------------------+


INFO  | Forward hook       | 72    | 0.010  | 0.005 | 0.330   | 0.3%   | transformer.h.11.mlp.c_proj                                                       |


INFO  +--------------------+-------+--------+-------+---------+--------+-----------------------------------------------------------------------------------+


INFO  | Capture inputs     | 1     | 0.037  | 0.037 | 0.037   | 0.0%   | cache_inputs:GPTNeoBlock                                                          |


INFO  +--------------------+-------+--------+-------+---------+--------+-----------------------------------------------------------------------------------+


In [30]:
from huggingface_hub import upload_folder,create_repo

create_repo(
    repo_id="moulee7788/gpt-neo-125M-4bit",  # name of your repo
    private=False,              # or True if you want it private
    exist_ok=True               # won't fail if it already exists
)

upload_folder(
    repo_id="moulee7788/gpt-neo-125M-4bit",  # username/repo_name
    folder_path=r"C:\Users\INMOR14\OneDrive - ABB\Documents\finetuning\EleutherAI\gpt-neo-125M-4bit",
    commit_message="Uploaded GPTQ quantized gpt-neo-125M model"
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/moulee7788/gpt-neo-125M-4bit/commit/7889024f7d4afd3a66c496acf998fd3e5b924dee', commit_message='Uploaded GPTQ quantized gpt-neo-125M model', commit_description='', oid='7889024f7d4afd3a66c496acf998fd3e5b924dee', pr_url=None, repo_url=RepoUrl('https://huggingface.co/moulee7788/gpt-neo-125M-4bit', endpoint='https://huggingface.co', repo_type='model', repo_id='moulee7788/gpt-neo-125M-4bit'), pr_revision=None, pr_num=None)